# Download CNN/DailyMail articles

- `data/raw/raw_news_stage1.jsonl`
- `data/processed/stage1_train.jsonl`
- `data/processed/stage1_validation.jsonl`

Stage-1 JSONL files.


## Import libraries and configure the download


In [ ]:
import json
import os
import random
import re
import time
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PARQUET_CACHE_DIR = RAW_DIR / "cnn_dailymail_parquet"

RAW_PATH = RAW_DIR / "raw_news_stage1.jsonl"
STAGE1_TRAIN_PATH = PROCESSED_DIR / "stage1_train.jsonl"
STAGE1_VAL_PATH = PROCESSED_DIR / "stage1_validation.jsonl"
STAGE2_SPLIT_PATHS = [
    PROCESSED_DIR / "train.jsonl",
    PROCESSED_DIR / "validation.jsonl",
    PROCESSED_DIR / "test.jsonl",
]

TRAIN_SAMPLE_SIZE = 5000
VAL_SAMPLE_SIZE = 500

MAX_ARTICLE_WORDS_EXCLUSIVE = 500
SEED = 67
OVERWRITE = True

# Data source
BASE_URL = "https://huggingface.co/datasets/abisee/cnn_dailymail/resolve/main/3.0.0"


TRAIN_PARQUET_URLS = [
    f"{BASE_URL}/train-00000-of-00003.parquet?download=true",
    f"{BASE_URL}/train-00001-of-00003.parquet?download=true",
    f"{BASE_URL}/train-00002-of-00003.parquet?download=true",
]

VALIDATION_PARQUET_URL = f"{BASE_URL}/validation-00000-of-00001.parquet?download=true"

WORD_RE = re.compile(r"\b\w+(?:[’'-]\w+)*\b", flags=re.UNICODE)

RAW_COLUMNS = [
    "article_id",
    "article",
    "summary",
    "article_word_count",
]

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

print(f"Parquet cache: {PARQUET_CACHE_DIR}")
print(f"Raw output: {RAW_PATH}")
print(f"Stage-1 train output: {STAGE1_TRAIN_PATH}")
print(f"Stage-1 validation output: {STAGE1_VAL_PATH}")
print(f"Training samples: {TRAIN_SAMPLE_SIZE}")
print(f"Validation samples: {VAL_SAMPLE_SIZE}")
print(f"Article filter: < {MAX_ARTICLE_WORDS_EXCLUSIVE} words")
print("Stage-1 records overlapping any Stage-2 split will be skipped and replaced.")


Parquet cache: data\raw\cnn_dailymail_parquet
Raw output: data\raw\raw_news_stage1.jsonl
Stage-1 train output: data\processed\stage1_train.jsonl
Stage-1 validation output: data\processed\stage1_validation.jsonl
Training samples: 5000
Validation samples: 500
Article filter: < 500 words
Stage-1 records overlapping any Stage-2 split will be skipped and replaced.


## Download and filter CNN/DailyMail


In [2]:
def count_words(text: str) -> int:
    return len(WORD_RE.findall(str(text)))


# Download files
def create_download_session():
    retry = Retry(total=8, connect=8, read=8, status=8, backoff_factor=2.0, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"], respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry)
    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": "Mozilla/5.0 (Stage1-News-Summary-Dataset)"})
    return session


def download_file(url: str, destination: Path):
    if (destination.exists() and destination.stat().st_size > 0):
        print(f"Using cached file: {destination}")
        return destination

    temporary_path = destination.with_suffix(destination.suffix + ".part")

    if temporary_path.exists():
        temporary_path.unlink()

    print(f"Downloading {destination.name} ...")

    session = create_download_session()

    try:
        with session.get(url, stream=True, timeout=(30, 300), allow_redirects=True) as response:
            response.raise_for_status()

            total_bytes = int(response.headers.get("content-length", 0))

            downloaded = 0
            next_report = 50 * 1024 * 1024

            with temporary_path.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                    if not chunk:
                        continue

                    handle.write(chunk)
                    downloaded += len(chunk)

                    if downloaded >= next_report:
                        if total_bytes > 0:
                            print(f"  {downloaded / 1024**2:.0f} MB / {total_bytes / 1024**2:.0f} MB")
                        else:
                            print(f"  {downloaded / 1024**2:.0f} MB")

                        next_report += 50 * 1024 * 1024

        if not temporary_path.exists() or temporary_path.stat().st_size == 0:
            raise RuntimeError("Empty download.")

        os.replace(temporary_path, destination)

        print(f"Saved: {destination} ({destination.stat().st_size / 1024**2:.1f} MB)")

        return destination

    finally:
        session.close()

        if temporary_path.exists() and temporary_path.stat().st_size == 0:
            temporary_path.unlink()


# Load Stage 2 exclusions
def load_stage2_exclusions(paths):
    excluded_ids = set()
    excluded_articles = set()

    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f"Missing file: {path}")

        with path.open('r', encoding='utf-8') as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue

                record = json.loads(line)
                article_id = str(record.get('article_id', '')).strip()
                article = str(record.get('article', '')).strip()

                if not article_id or not article:
                    raise ValueError(f"Invalid record: {path}:{line_number}")

                excluded_ids.add(article_id)
                excluded_articles.add(article)

    return excluded_ids, excluded_articles


STAGE2_EXCLUDED_IDS, STAGE2_EXCLUDED_ARTICLES = load_stage2_exclusions(STAGE2_SPLIT_PATHS)

print(f"Loaded {len(STAGE2_EXCLUDED_IDS)} Stage-2 IDs to exclude from Stage 1.")


# Read Parquet records
def records_from_parquet(parquet_path: Path):
    frame = pd.read_parquet(parquet_path, columns=["id", "article", "highlights"])

    qualifying = []
    seen_ids = set()
    seen_articles = set()

    for row in frame.itertuples(index=False):
        article_id = str(row.id).strip()
        article = str(row.article).strip()
        summary = str(row.highlights).strip()

        if not article_id or not article or not summary:
            continue

        if article_id in STAGE2_EXCLUDED_IDS or article in STAGE2_EXCLUDED_ARTICLES or article_id in seen_ids or article in seen_articles:
            continue

        article_word_count = count_words(article)

        if article_word_count >= MAX_ARTICLE_WORDS_EXCLUSIVE:
            continue

        qualifying.append({"article_id": article_id, "article": article, "summary": summary, "article_word_count": article_word_count})
        seen_ids.add(article_id)
        seen_articles.add(article)

    return qualifying


# Stage 1 training data
train_pool = []

for shard_index, url in enumerate(TRAIN_PARQUET_URLS):
    shard_path = PARQUET_CACHE_DIR / f"train-{shard_index:05d}.parquet"
    download_file(url, shard_path)
    shard_records = records_from_parquet(shard_path)

    existing_train_ids = {record['article_id'] for record in train_pool}    
    existing_train_articles = {record['article'] for record in train_pool}
    shard_records = [
        record for record in shard_records
        if record['article_id'] not in existing_train_ids
        and record['article'] not in existing_train_articles
    ]

    train_pool.extend(shard_records)

    print(f"Train shard {shard_index}: {len(shard_records)} qualifying; pool={len(train_pool)}")

    if (len(train_pool) >= TRAIN_SAMPLE_SIZE):
        break


if len(train_pool) < TRAIN_SAMPLE_SIZE:
    raise RuntimeError(f"Need {TRAIN_SAMPLE_SIZE} training records; found {len(train_pool)}.")


random.Random(SEED).shuffle(train_pool)
stage1_train_records = train_pool[:TRAIN_SAMPLE_SIZE]


# Stage 1 validation data
validation_path = PARQUET_CACHE_DIR / "validation.parquet"

download_file(VALIDATION_PARQUET_URL, validation_path)

validation_pool = records_from_parquet(validation_path)
selected_train_ids = {record['article_id'] for record in stage1_train_records}
selected_train_articles = {record['article'] for record in stage1_train_records}
validation_pool = [
    record for record in validation_pool
    if record['article_id'] not in selected_train_ids
    and record['article'] not in selected_train_articles
]

if len(validation_pool) < VAL_SAMPLE_SIZE:
    raise RuntimeError(f"Need {VAL_SAMPLE_SIZE} validation records; found {len(validation_pool)}.")

random.Random(SEED + 1).shuffle(validation_pool)
stage1_val_records = validation_pool[:VAL_SAMPLE_SIZE]
records = stage1_train_records + stage1_val_records

print("\nPrepared Stage-1 training:", len(stage1_train_records))
print("Prepared Stage-1 validation:", len(stage1_val_records))


Loaded 500 Stage-2 IDs to exclude from Stage 1.
Using cached file: data\raw\cnn_dailymail_parquet\train-00000.parquet
Train shard 0: 34814 qualifying; pool=34814
Saved: data\raw\cnn_dailymail_parquet\validation.parquet (33.1 MB)

Prepared Stage-1 training: 5000
Prepared Stage-1 validation: 500


## Save the raw dataset

In [3]:
# Save dataset
def write_jsonl_atomic(path: Path, records_to_write: list):
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f"File exists: {path}")

    temporary_path = path.with_suffix(path.suffix + ".tmp")

    try:
        with temporary_path.open("w", encoding="utf-8", newline="\n") as handle:
            for record in records_to_write:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")

        os.replace(temporary_path, path)

    finally:
        if temporary_path.exists():
            temporary_path.unlink()


train_ids = {record["article_id"] for record in stage1_train_records}
val_ids = {record["article_id"] for record in stage1_val_records}
overlap = train_ids & val_ids

if overlap:
    raise ValueError("Training and validation sets overlap.")

stage1_ids = train_ids | val_ids
cross_stage_overlap = stage1_ids & STAGE2_EXCLUDED_IDS

if cross_stage_overlap:
    raise ValueError("Stage 1 and Stage 2 overlap.")


write_jsonl_atomic(RAW_PATH, records)
write_jsonl_atomic(STAGE1_TRAIN_PATH, stage1_train_records)
write_jsonl_atomic(STAGE1_VAL_PATH, stage1_val_records)


all_word_counts = [record["article_word_count"] for record in stage1_train_records + stage1_val_records]

print(f"Saved {len(stage1_train_records)} Stage-1 training records to '{STAGE1_TRAIN_PATH}'")
print(f"Saved {len(stage1_val_records)} Stage-1 validation records to '{STAGE1_VAL_PATH}'")
print(f"Word-count range: {min(all_word_counts)} to {max(all_word_counts)}")


Saved 5000 Stage-1 training records to 'data\processed\stage1_train.jsonl'
Saved 500 Stage-1 validation records to 'data\processed\stage1_validation.jsonl'
Word-count range: 14 to 499
